# Mini Project 1: Scraping Data from a Dynamic Webpage

## Setup (Colab only - skip this cell if you run it locally)

In [ ]:
!pip install selenium beautifulsoup4 chromedriver-autoinstaller -q
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb

In [ ]:
import re
import time

import pandas as pd
import chromedriver_autoinstaller
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## Initialize the WebDriver

In [ ]:
chromedriver_autoinstaller.install()

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

## Load the page

In [ ]:
url = "https://www.inmotionhosting.com/shared-hosting"
driver.get(url)

# wait until the plan cards are rendered
WebDriverWait(driver, 20).until(
    EC.presence_of_element_located((By.XPATH, "//h3[normalize-space()='Launch']"))
)
driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 3)")
time.sleep(2)

html = driver.page_source
print(driver.title)
print(len(html))

## Extract plan names, features and prices

In [ ]:
soup = BeautifulSoup(html, "html.parser")

plan_names = ["Launch", "Power", "Pro"]
terms = ["3 years", "2 years", "1 year", "1 month"]


def count_plans(tag):
    return len([h for h in tag.find_all("h3") if h.get_text(strip=True) in plan_names])


plans = []
for name in plan_names:
    heading = soup.find("h3", string=lambda s: s and s.strip() == name)
    if heading is None:
        print("plan not found:", name)
        continue

    # go up until the next level would contain more than one plan
    card = heading
    while card.parent is not None and count_plans(card.parent) == 1:
        card = card.parent

    text = card.get_text(" ", strip=True)
    prices = re.findall(r"\$\d+\.\d+/mo", text)
    features = [li.get_text(" ", strip=True) for li in card.find_all("li")]

    row = {"plan": name, "features": " | ".join(features)}
    for term, price in zip(terms, prices[0::2]):
        row["price " + term] = price
    row["renewal price (3 years)"] = prices[1] if len(prices) > 1 else None
    plans.append(row)

print(len(plans), "plans found")

## Store and save the data

In [ ]:
df = pd.DataFrame(plans)
print(df)
df.to_csv("inmotion_plans.csv", index=False)
print("saved inmotion_plans.csv")

In [ ]:
# Colab only
from google.colab import files
files.download("inmotion_plans.csv")

## Close the WebDriver

In [ ]:
driver.quit()